# ROGII Wellbore Noobnote: TVT and GR Alignment

This note is an English version of my local ROGII Wellbore explanation page.

It is not a final solution and not a public-notebook review yet. It is a beginner-facing note for understanding the competition problem before reading solution code.

## Questions First

Before reading any model notebook, these were the questions I needed to answer:

- If the well is drilled horizontally, why is the target called `True Vertical Thickness`?
- What exactly is hidden in the rows where `TVT_input` is `NaN`?
- Why do many approaches compare `GR` curves instead of just joining two tables?
- What is a `typewell`, and why does it help a horizontal well problem?
- What do terms such as `NCC`, `DWT`, `Ridge`, and `Hill Climbing` mean in plain language?
- Why can a good leaderboard score still be risky?

This note answers those questions first. After that, reading public notebooks becomes much less confusing.

## Competition In One Sentence

For each well, we read CSV files and predict the hidden geological position, row by row, in the interval where the answer is missing.

## What Is TVT?

`TVT` means `True Vertical Thickness`.

In this competition, TVT is the value we need to predict. It represents where each point of the horizontal well sits inside the vertical geological reference frame.

At first this feels strange: why do we care about a "vertical" value when the well is drilled horizontally?

The reason is that horizontal drilling is usually trying to stay inside a target reservoir layer. That layer can be very thin. If the drill path moves slightly too high or too low, it can leave the productive layer.

Geological layers are not perfectly flat. They can tilt, bend, or wave. So even if the drill keeps moving horizontally, its position relative to the target layer can drift upward or downward.

TVT is the number that helps describe that relative geological position. If we can track TVT, we can tell whether the well is still inside the desired layer or is drifting away from it.

In short: horizontal drilling makes TVT important because we need to keep asking, "Are we still inside the target layer?"

## Horizontal Wells and Geology

The goal of a horizontal well is to move along the productive layer for as long as possible.

If the well leaves that layer, production can drop. The hard part is that the layer is not a perfectly horizontal sheet. It may be slanted or wavy.

A vertical well is easier to interpret:

- As depth increases, the well passes through one geological layer after another.
- The `GR` log can often show which layer the well is currently passing through.

A horizontal well is harder:

- The vertical depth may change only slowly.
- The well moves forward through space.
- Changes in `GR` might mean the well is crossing geological structure, or they might simply reflect variation inside the same layer.

The common strategy is to compare the `GR` curve from the horizontal well with a reference `GR` curve from a vertical well.

## The Role of the Typewell

Each well has a paired `typewell.csv` file.

A typewell is a vertical reference well. Because it is vertical, the relation between depth and geological position is easier to interpret. It contains `TVT`, `GR`, and geological labels.

The main idea is:

1. The typewell has a known vertical reference curve.
2. The horizontal well has a `GR` curve measured while drilling forward.
3. If a short segment of the horizontal `GR` curve has the same shape as a segment of the typewell `GR` curve, those two segments may correspond to the same geological position.
4. We can then borrow the typewell `TVT` value at that matching position and use it as a candidate TVT for the horizontal well.

This is not a SQL-style join. There is no shared key column that directly tells us which row matches which row.

It is a shape-matching problem.

## What "Matching" Means Here

The matching operation is closer to searching for the same melody in two recordings than joining two tables by ID.

A typical process is:

1. Cut a short window from the horizontal well `GR` curve, for example 15 rows before and after the current point.
2. Slide that window across the typewell `GR` curve.
3. At each position, compute how similar the two shapes are.
4. Choose the typewell position with the highest similarity.
5. Use the `TVT` value at that typewell position as the candidate TVT for the horizontal well row.

The similarity score is often computed with `NCC`, which stands for normalized cross-correlation.

NCC is a way to compare two local curve shapes after normalizing their scale. In plain language, it asks: "Do these two short curve segments go up and down in the same pattern?"

## Data and Terms

Each well has a `horizontal_well.csv` file.

Each row corresponds to one point along the well path. The table is ordered by `MD`, which means measured depth. `MD` is distance measured along the well path, not straight vertical depth.

Important columns:

- `WELLNAME`: well ID.
- `MD`: measured depth along the well path.
- `X`, `Y`, `Z`: 3D coordinates. `X` and `Y` are map position. `Z` is vertical depth.
- `GR`: gamma ray measurement.
- `TVT`: the true target value in training data.
- `TVT_input`: a visible copy of TVT where the target is known. It becomes `NaN` in the prediction interval.
- `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, `BUDA`: formation surface columns.

The submitted file must contain:

- `id`: row identifier from `sample_submission.csv`.
- `tvt`: predicted TVT value.

The rows where `TVT_input` is `NaN` are the rows we need to predict. A consecutive hidden region is often called the evaluation zone.

## GR As A Geological Signal

`GR` means gamma ray.

It measures natural radioactivity from rocks along the well path. Different rock types tend to produce different gamma ray levels. Shale and clay-rich rocks usually produce higher values. Cleaner sands or carbonates often produce lower values.

If we plot `GR` along the well path, we get a curve with peaks and valleys. That curve is useful because the pattern often reflects the sequence of geological layers.

The key idea is that the shape of the `GR` curve can act like a geological fingerprint.

## Formation Surfaces

Columns such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, and `BUDA` describe formation surfaces.

You can think of them as estimated depths of important geological boundaries. They help answer questions like:

- How far is this row from the top of a certain formation?
- Is the well likely above, inside, or below a known layer?
- Does the `GR` alignment agree with the larger geological structure?

`GR` shape matching can be noisy, so formation surfaces are useful as broader geological context.

## DWT

`DWT` means discrete wavelet transform.

For this competition, DWT is a way to turn a `GR` curve into model-friendly numerical features.

A raw `GR` curve can contain both large-scale trends and small local wiggles. DWT decomposes the curve into components at different scales:

- large-scale waves over longer distances;
- smaller local bumps and dips;
- fine-grained local irregularities.

After that decomposition, models such as Ridge, LightGBM, or CatBoost can use the extracted values as features.

In beginner terms: DWT converts a curve shape into columns that a tabular model can learn from.

## Ridge, LightGBM, CatBoost, and Hill Climbing

`Ridge` is a regularized linear regression model. It is simple, stable, and useful when many related features are available.

`LightGBM` and `CatBoost` are tree-based machine learning models. They can learn nonlinear relations between features such as `MD`, `X/Y/Z`, `GR`, formation surfaces, and alignment signals.

`Hill Climbing` is usually used as a post-processing or optimization step. It starts from an existing prediction and repeatedly tries small changes:

1. Increase or decrease one predicted value slightly.
2. Keep the change if the score improves.
3. Revert it if the score gets worse.
4. Repeat across many rows.

For wells, this can help because neighboring rows should usually change smoothly. A very jagged TVT curve can be geologically unnatural.

## LB, CV, and Leakage Risk

`LB` means leaderboard score on Kaggle.

`CV` means local cross-validation score.

In ROGII, train and test wells can differ. A method that looks strong on public LB may be tuned too much to visible test behavior or public artifacts. That is why CV design matters.

A key rule is to split by `WELLNAME`. Rows from the same well should not be split across train and validation folds. If they are mixed, the validation score can be too optimistic because information from the same well leaks across folds.

## What To Read Next

After this note, the next useful step is to read public notebooks by role, not by title.

Start with EDA notebooks that show the files, columns, missing target interval, and basic plots.

Then read notebooks about `GR` alignment and `typewell` matching.

After that, read modeling pipelines using Ridge, DWT features, LightGBM, CatBoost, or similar tabular models.

Finally, read post-processing and blending notebooks, while keeping overfitting and leaderboard risk in mind.

## Scope Of This Note

This note is intentionally a competition-understanding note, not a notebook-by-notebook review.

It explains the core problem first:

- what TVT means;
- why a horizontal well still needs a vertical geological reference;
- how `GR` can be used as a geological signal;
- why `typewell` matching is closer to shape matching than table joining;
- why terms such as `NCC`, `DWT`, `Ridge`, and `Hill Climbing` appear in many ROGII solutions.

At this stage, link lists are less important than building the mental model. The reader should be able to open a public notebook afterward and understand why the code is talking about `TVT_input`, `GR` windows, typewell alignment, formation surfaces, smoothing, CV, and LB risk.

## What Should Come Next

The next note should be a true public-notebook reading note. That should go notebook by notebook and record:

- what the notebook is trying to do;
- what model or signal it mainly relies on;
- what score or claim it reports;
- what implementation blocks look reusable;
- what looks risky, overfit, or dependent on public artifacts;
- whether it is useful for EDA, feature engineering, modeling, post-processing, or blending.

For this introductory note, the main job is simpler: make the ROGII problem understandable before reading those notebooks.
